# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hafsaShaban/flyrank_internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
if not os.path.exists('/content/flyrank_internship'):
    !git clone https://github.com/hafsaShaban/flyrank_internship.git
%cd /content/flyrank_internship

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print("Loaded:", len(df), "rows")

/content/flyrank_internship
Loaded: 30000 rows


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
numeric_features = ['word_count','char_count','ctr','avg_position','engagement_rate','scroll_rate',
    'ai_traffic_pct','content_age_days','days_since_last_update','search_volume','cpc',
    'impressions_90d','clicks_90d','pageviews_90d','sessions_90d','users_90d',
    'engaged_sessions_90d','ai_sessions_90d','scroll_events_90d',
    'days_with_impressions','days_with_sessions']
categorical_features = ['content_type','main_intent','competition_level','freshness_tier',
    'word_count_tier','char_count_tier','impression_tier','position_tier']

X = df[numeric_features].copy()
for col in numeric_features:
    X[f'has_{col}'] = X[col].notnull().astype(int)
    X[col] = X[col].fillna(0)

X_cat = pd.get_dummies(df[categorical_features], dummy_na=True)
X = pd.concat([X, X_cat], axis=1)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (30000, 81)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [4]:
notes = pd.DataFrame({
    'feature': numeric_features,
    'pct_missing': [df[c].isnull().mean() for c in numeric_features],
    'available_before_prediction': True  # all pulled from ML-04's feature bucket
})
print(notes)

                   feature  pct_missing  available_before_prediction
0               word_count     0.256633                         True
1               char_count     0.256633                         True
2                      ctr     0.000000                         True
3             avg_position     0.000000                         True
4          engagement_rate     0.000000                         True
5              scroll_rate     0.004167                         True
6           ai_traffic_pct     0.000000                         True
7         content_age_days     0.000000                         True
8   days_since_last_update     0.000000                         True
9            search_volume     0.082267                         True
10                     cpc     0.082267                         True
11         impressions_90d     0.000000                         True
12              clicks_90d     0.000000                         True
13           pageviews_90d     0.0

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [5]:
groups = df['client_id']
gkf = GroupKFold(n_splits=5)

def grouped_auc(X_input, y, groups):
    scores = []
    for train_idx, test_idx in gkf.split(X_input, y, groups):
        model = RandomForestClassifier(n_estimators=100, random_state=42)
        model.fit(X_input.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict_proba(X_input.iloc[test_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[test_idx], preds))
    return sum(scores) / len(scores)

y = df['is_declining_label']

auc_with = grouped_auc(X, y, groups)
print("Grouped AUC WITH all current features:", auc_with)

base_rate = y.mean()
print("Base rate (declining %):", base_rate)

# Sanity check: confirm trend_direction / trend_pct never made it into X
leaky_check = [c for c in X.columns if 'trend' in c.lower()]
print("Any trend_* columns accidentally in features?:", leaky_check)


Grouped AUC WITH all current features: 0.6602736373481104
Base rate (declining %): 0.5420666666666667
Any trend_* columns accidentally in features?: []


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [6]:
excluded = {
    'trend_direction': 'label-derived — label is computed from this',
    'trend_pct': 'label-derived — label is computed from this',
    'impressions_last_30d': 'overlaps label window, leakage risk',
    'clicks_last_30d': 'overlaps label window, leakage risk',
    'sessions_last_30d': 'overlaps label window, leakage risk',
    'provider_used': 'decision-derived product flag, not a signal',
    'model_used': 'decision-derived product flag, not a signal',
    'content_id': 'pseudonymous ID, context only',
    'client_id': 'pseudonymous ID, context only (used for grouped split)',
}
for col, reason in excluded.items():
    print(f"{col}: {reason}")

still_present = [c for c in excluded if c in X.columns]
print("\nExcluded columns accidentally still in feature matrix:", still_present)


trend_direction: label-derived — label is computed from this
trend_pct: label-derived — label is computed from this
impressions_last_30d: overlaps label window, leakage risk
clicks_last_30d: overlaps label window, leakage risk
sessions_last_30d: overlaps label window, leakage risk
provider_used: decision-derived product flag, not a signal
model_used: decision-derived product flag, not a signal
content_id: pseudonymous ID, context only
client_id: pseudonymous ID, context only (used for grouped split)

Excluded columns accidentally still in feature matrix: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.